In [1]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer,Trainer,TrainingArguments
from transformers import DataCollatorForLanguageModeling
import torch
from torch.utils.data import Dataset

In [2]:
domain_corpus = [
    "A stock market is a public market for the trading of company stock and derivatives at an agreed price. It is a key component of the financial system.",
    "Inflation refers to the rate at which the general level of prices for goods and services is rising, and subsequently, purchasing power is falling.",
    "A bond is a fixed-income instrument that represents a loan made by an investor to a borrower, typically corporate or governmental.",
    "The gross domestic product (GDP) is the total value of everything produced by all the people and companies in the country.",
    "Interest rates are the cost of borrowing money or the return for investing money. Central banks influence interest rates to control inflation and stabilize the currency.",
    "An exchange-traded fund (ETF) is a type of investment fund and exchange-traded product, meaning they are traded on stock exchanges.",
    "A mutual fund is a professionally managed investment fund that pools money from many investors to purchase securities.",
    "Risk management in finance involves identifying, assessing, and prioritizing risks followed by coordinated efforts to minimize, monitor, and control the probability or impact of events.",
    "Corporate finance deals with the capital structure of corporations, including the funding and the actions that management takes to increase the value of the company.",
    "Quantitative easing is a monetary policy whereby a central bank buys government securities or other securities from the market to lower interest rates and increase the money supply."
]

In [3]:
class TextDataset(Dataset):
    def __init__(self, texts,tokenizer,block_size=120):
        self.examples = []
        for text in texts:
            self.examples.append(tokenizer(text,truncation=True,padding='max_length',max_length=block_size,return_tensors="pt"))


    def __len__(self):
      return len(self.examples)
    def __getitem__(self,i):
      return {key:val.squeeze() for key,val in self.examples[i].items()}

In [4]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.generation_config.pd_token_id = tokenizer.eos_token_id

train_dataset = TextDataset(domain_corpus,tokenizer)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
training_srgs = TrainingArguments(
    output_dir = './results',
    overwrite_output_dir=True,
    num_train_epochs = 3,
    per_device_train_batch_size =4,
    save_steps =10_000,
    save_total_limit =2,)

trainer = Trainer(
    model=model,
    args=training_srgs,
    data_collator=data_collator,
    train_dataset=train_dataset,
    #eval_dataset=test_dataset Add if you have a validation/text dataset
)

#fine tune the model
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jas1420047 (jas1420047-mn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


TrainOutput(global_step=9, training_loss=2.5926886664496527, metrics={'train_runtime': 231.1576, 'train_samples_per_second': 0.13, 'train_steps_per_second': 0.039, 'total_flos': 1837209600000.0, 'train_loss': 2.5926886664496527, 'epoch': 3.0})

In [9]:
def generate_response_domain_adaption(query,model,tokenizer):
  inputs = tokenizer.encode_plus(query,return_tensors ='pt', return_attention_mask=True)
  input_ids = inputs['input_ids']
  attention_mask = inputs['attention_mask']

  output = model.generate(input_ids, attention_mask=attention_mask, max_length=200, num_return_sequences =1, pad_token_id=tokenizer.eos_token_id)
  response = tokenizer.decode(output[0],skip_special_tokens=True)
  return response

#Sample query
query = "What is a stock market?"
response = generate_response_domain_adaption(query,model,tokenizer)
print(response)

What is a stock market?
A stock market is a market for stocks and bonds. It is a market where investors buy and sell securities.

A stock is a stock that is traded on the market.
A stock is a stock that is traded on the market.

A is a stock that is traded on the market.

A is a stock that is traded on the market.

A is a stock that is traded on the market.
A is a stock that is traded on the market.
A is a stock that is traded on the market.

A is a stock that is traded on the market.

A is a stock that is traded on the market.

A is a stock that is traded on the market.

A is a stock that is traded on the market.
A is a stock that is traded on the market.

A is a stock that is traded on the market.

A is
